# 31_deeper_C — Shallow nonlinear Stage-C gate ablation

Paired three-seed comparison of the exact Notebook 31 compact soft MoE against a gate-only challenger. The encoder remains `47 → 128 → 64`, all four experts remain linear `64 → 22`, and only the gate changes from `64 → 4` to dropout-free `64 → 32 → 4` with ReLU. Exact baseline Stage-A and Stage-B checkpoints are reused, so the challenger trains only Stage C.

This immutable gate-only contract retains Notebook 31's Stage-C learning rate of `0.001`. If Notebook 31c later establishes `0.0003` as the better rate, test that gate × learning-rate combination under a new prefix rather than changing this result after the fact.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDIES = [
    ('baseline_linear_gate', 'config/compact_soft_moe_3seed.yaml', 'nfv3_4way_moe_soft_comparable_repro_v1'),
    ('deeper_gate32', 'config/compact_soft_moe_deeper_gate_3seed.yaml', 'nfv3_4way_moe_soft_deeper_gate32_v1'),
]
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS_PER_STUDY = 3
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO
url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None
os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR
os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Contract preflight, dry-run, and resumable execution

The baseline is completed first if necessary. For every paired seed, the challenger loads the baseline's exact Stage-A encoder and Stage-B expert bank, initializes only the `64 → 32 → 4` gate, and retrains Stage C. A changed topology or learning rate requires a new prefix.

In [ ]:
import torch, yaml
from training.logging_utils import run_streaming_logged
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before executing the study.')
gate_study = yaml.safe_load(Path('config/compact_soft_moe_deeper_gate_3seed.yaml').read_text())
assert gate_study['seeds'] == [0, 1, 2]
assert gate_study['reuse_stage_a_from_prefix'] == 'nfv3_4way_moe_soft_comparable_repro_v1'
assert gate_study['reuse_stage_b_from_prefix'] == 'nfv3_4way_moe_soft_comparable_repro_v1'
assert gate_study['backbone']['model']['encoder']['hidden_dims'] == [128]
assert gate_study['backbone']['model']['expert']['hidden_dims'] == []
assert gate_study['backbone']['model']['gate']['hidden_dims'] == [32]
assert gate_study['backbone']['training']['stage_c']['optimizer']['lr'] == 0.001
assert gate_study['expected_total_parameters'] == 22332
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py', 'tests/test_training_observability.py'], check=True)
for condition, study_config, prefix in STUDIES:
    study = yaml.safe_load(Path(study_config).read_text())
    summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary'
    summary_dir.mkdir(parents=True, exist_ok=True)
    gate_dims = study['backbone']['model']['gate']['hidden_dims']
    print(f'=== {condition}: gate hidden={gate_dims}, expected parameters={study["expected_total_parameters"]:,} ===')
    command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', study_config, '--prefix', prefix]
    run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-deeper-C-{condition}-dry-run')
    if EXECUTE:
        launch = [*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS_PER_STUDY)]
        run_streaming_logged(launch, str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-deeper-C-{condition}-training')
if not EXECUTE:
    print('NO TRAINING WAS STARTED. Review the contracts, set EXECUTE=True, and rerun this cell.')

## Paired validation, locked-test, per-class, and resource comparison

In [ ]:
import pandas as pd
from IPython.display import Image, display
summaries = {condition: Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' for condition, _, prefix in STUDIES}
complete = {name: (path / 'Final_Study_Manifest.json').is_file() for name, path in summaries.items()}
print('Completion:', complete)
if all(complete.values()):
    validation = []; test = []; classes = []; resources = []
    for name, path in summaries.items():
        for filename, target in [('Three_Seed_Guardrails.csv', validation), ('Three_Seed_Overall.csv', test), ('Three_Seed_Per_Class.csv', classes), ('Three_Seed_Resource_Accounting.csv', resources)]:
            frame = pd.read_csv(path / filename); frame.insert(0, 'condition', name); target.append(frame)
    validation = pd.concat(validation, ignore_index=True)
    test = pd.concat(test, ignore_index=True)
    print('=== Primary validation comparison ===')
    display(validation[['condition', 'seed', 'validation_macro_f1', 'worst_dataset_macro_f1', 'rare_recall_mean']])
    pivot = validation.pivot(index='seed', columns='condition', values='validation_macro_f1')
    pivot['delta_deeper_gate_minus_linear'] = pivot['deeper_gate32'] - pivot['baseline_linear_gate']
    display(pivot)
    metrics = [c for c in ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'weighted_f1', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
    print('=== Locked-test descriptive mean and SD ===')
    display(test.groupby('condition')[metrics].agg(['mean', 'std']))
    print('=== Per-class F1 mean ===')
    display(pd.concat(classes, ignore_index=True).pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    print('=== Resource accounting ===')
    display(pd.concat(resources, ignore_index=True))
else:
    print('Comparison appears after both three-seed manifests are complete.')

## Gate behavior and Stage-B competence preservation

The standard reports provide mean gate weights by source dataset and hard-argmax expert utilization. The derived entropies below describe those aggregate distributions; they detect expert domination but are not substitutes for mean per-sample gate entropy. The final table compares each Stage-C expert against the exact reused Stage-B baseline expert bank.

In [ ]:
if all(complete.values()):
    import numpy as np
    prefixes = {condition: prefix for condition, _, prefix in STUDIES}
    routing_rows = []; utilization_rows = []; preservation_rows = []
    baseline_prefix = prefixes['baseline_linear_gate']
    for condition, prefix in prefixes.items():
        for seed in (0, 1, 2):
            result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
            gate = pd.read_csv(result_dir / 'Gate_By_Dataset.csv')
            gate_columns = [column for column in gate if column.startswith('mean_gate__')]
            probabilities = gate[gate_columns].to_numpy(float)
            probabilities /= probabilities.sum(axis=1, keepdims=True)
            entropy = -(probabilities * np.log(np.clip(probabilities, 1e-12, 1.0))).sum(axis=1)
            for row_index, row in gate.reset_index(drop=True).iterrows():
                routing_rows.append({'condition': condition, 'seed': seed, 'origin': row['origin'], 'entropy_of_mean_gate': entropy[row_index], 'max_mean_gate_weight': probabilities[row_index].max(), **{column: row[column] for column in gate_columns}})
            utilization = pd.read_csv(result_dir / 'Expert_Utilization.csv')
            fractions = utilization['hard_argmax_fraction'].to_numpy(float)
            utilization_rows.append({'condition': condition, 'seed': seed, 'utilization_entropy': -(fractions * np.log(np.clip(fractions, 1e-12, 1.0))).sum(), 'largest_expert_fraction': fractions.max()})
            stage_b = pd.read_csv(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{baseline_prefix}_seed{seed}' / 'Validation_Expert_Owned_StageB.csv')
            stage_c = pd.read_csv(result_dir / 'Validation_Expert_Owned_StageC.csv')
            merged = stage_b[['dataset', 'macro_f1']].merge(stage_c[['dataset', 'macro_f1']], on='dataset', suffixes=('_stage_b', '_stage_c'))
            merged.insert(0, 'seed', seed); merged.insert(0, 'condition', condition)
            merged['stage_c_minus_reused_stage_b'] = merged['macro_f1_stage_c'] - merged['macro_f1_stage_b']
            preservation_rows.append(merged)
    print('=== Routing by source dataset ==='); display(pd.DataFrame(routing_rows))
    print('=== Expert utilization collapse checks ==='); display(pd.DataFrame(utilization_rows))
    print('=== Owned-expert competence after Stage C ==='); display(pd.concat(preservation_rows, ignore_index=True))

## Stage-C training stability and latent-space comparison

In [ ]:
if all(complete.values()):
    histories = []; latent = []
    for name, path in summaries.items():
        frame = pd.read_csv(path / 'Training_History_3Seed.csv'); frame.insert(0, 'condition', name); histories.append(frame)
        frame = pd.read_csv(path / 'Latent_Snapshot_Summary_3Seed.csv'); frame.insert(0, 'condition', name); latent.append(frame)
    stage_c = pd.concat(histories, ignore_index=True).query("stage == 'C'")
    display(stage_c[['condition', 'seed', 'epoch', 'learning_rate', 'train_total_loss', 'train_ce_loss', 'train_balance_penalty', 'val_macro_f1', 'best_val_macro_f1', 'patience_left']])
    display(pd.concat(latent, ignore_index=True))
    for name, path in summaries.items():
        print('===', name, 'training figures ===')
        for figure in sorted((path / 'training_figures').glob('*.png')):
            display(Image(filename=str(figure)))